In [ ]:
!pip install openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 18.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=4ca9813f1cddb7d5ce2861252a38fc430f27af5763c5c64ea82cb2d2ba105736
  Stored in directory: /root/.cache/pip/wheels/ca/58/d5/fb4539ad74c3ca81eb40f7eda020ac77d080b33ad57449d485
Successfully built openai-whisper


In [ ]:
import whisper
model_whisper = whisper.load_model("base")

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving TTSOL-en-AU-Natasha-20260925-040951.mp3 to TTSOL-en-AU-Natasha-20260925-040951.mp3


In [ ]:
audio_file = list(uploaded.keys())[0]

result = model_whisper.transcribe(audio_file)

print(result["text"])

 Hello, my name is Karim. I am Learning Artificial Intelligence and Machine Learning.


In [ ]:
!pip install transformers accelerate

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
prompt = "What is machine learning?"

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

What is machine learning? Machine learning is a type of artificial intelligence that enables computers to learn from data without being explicitly programmed. It involves using algorithms and statistical models to allow machines to automatically improve their performance based on new input or feedback.
Machine learning can be applied in many different fields, including natural language processing, computer vision, speech recognition, recommendation systems, fraud detection, and more. By training the system with large amounts of data, it can learn patterns and relationships within the data, allowing it to make predictions or decisions based


In [ ]:
!pip install gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.5.0
    Uninstalling click-8.5.0:
      Successfully uninstalled click-8.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
huggingface-hub 1.29.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.


In [ ]:
messages = [
    {
        "role": "user",
        "content": "What is machine learning?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",
    add_generation_prompt=True
)

input_ids = inputs["input_ids"]

with torch.no_grad():
    outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=50
    )

response = tokenizer.decode(
    outputs[0][input_ids.shape[-1]:],
    skip_special_tokens=True
)

print(response)

Machine learning is a subset of artificial intelligence that involves the development of algorithms and statistical models that enable computers to learn from and make predictions or decisions without being explicitly programmed. In other words, it's about teaching machines to improve their performance based on data.




In [ ]:
def voice_assistant(audio_file):

    # 1. Speech-to-Text
    result = model_whisper.transcribe(audio_file)
    text = result["text"].strip()

    # 2. LLM
    messages = [
        {
            "role": "user",
            "content": text
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    )

    input_ids = inputs["input_ids"]

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            max_new_tokens=50
        )

    response = tokenizer.decode(
        outputs[0][input_ids.shape[-1]:],
        skip_special_tokens=True
    )

    # 3. TTS
    tts = gTTS(text=response, lang="en")
    tts.save("response.mp3")

    return text, response, "response.mp3"

In [ ]:
from google.colab import files

uploaded = files.upload()
audio_file = list(uploaded.keys())[0]

Saving TTSOL-en-AU-Natasha-20260925-043342.mp3 to TTSOL-en-AU-Natasha-20260925-043342 (4).mp3


In [ ]:
text, response, audio_file = voice_assistant(audio_file)

print("User:", text)
print("Assistant:", response)
print("Audio:", audio_file)

User: What is machine learning?
Assistant: Machine Learning (ML) is a type of artificial intelligence that allows computers to learn from data without being explicitly programmed. It involves using algorithms and statistical models to enable systems to improve their performance on a specific task over time.

In simpler terms, machine learning
Audio: response.mp3


In [ ]:
from IPython.display import Audio, display

display(Audio("response.mp3"))

In [ ]:
import gradio as gr

def run_voice_assistant(audio_file):

    text, response, audio_output = voice_assistant(audio_file)

    return text, response, audio_output


with gr.Blocks() as demo:

    gr.Markdown("# 🎙️ Voice Assistant")
    gr.Markdown(
        "Speak to the assistant and get a text and voice response."
    )

    audio_input = gr.Audio(
        sources=["microphone", "upload"],
        type="filepath",
        label="Your Voice"
    )

    button = gr.Button("Ask Assistant")

    transcription = gr.Textbox(
        label="Your Speech"
    )

    response = gr.Textbox(
        label="Assistant Response"
    )

    audio_output = gr.Audio(
        label="Assistant Voice"
    )

    button.click(
        fn=run_voice_assistant,
        inputs=audio_input,
        outputs=[
            transcription,
            response,
            audio_output
        ]
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://494cb692463b9bbdfe.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
